# Nederlands LLM Datasets Verkennen

Dit notebook laadt en verkent alle Nederlandse datasets uit de `datasets/` folder.

In [1]:
# Import libraries
from datasets import load_from_disk, load_dataset
import pandas as pd
from pathlib import Path
import os

# Define dataset path
DATASET_DIR = Path('.')

print(f"Datasets in: {DATASET_DIR}")
print(f"Available datasets:")
for item in sorted(DATASET_DIR.iterdir()):
    if item.is_dir() and not item.name.startswith('.'):
        print(f"  - {item.name}")

Datasets in: .
Available datasets:
  - squad_nl
  - wikipedia_nl
  - wikipedia_nl_sample_processed
  - wikipedia_nl_tokenized


## 1. Wikipedia NL

In [ ]:
# Load Wikipedia NL
wiki_nl = load_from_disk('wikipedia_nl')

print("=" * 60)
print("WIKIPEDIA NL")
print("=" * 60)
print(f"Total samples: {len(wiki_nl):,}")
print(f"Columns: {wiki_nl.column_names}")
print(f"Dataset size: ~2.5 GB")
print()
print("Sample article:")
print(f"ID: {wiki_nl[0]['id']}")
print(f"Title: {wiki_nl[0]['title']}")
print(f"URL: {wiki_nl[0]['url']}")
print(f"Text preview (first 300 chars):\n{wiki_nl[0]['text'][:300]}...")

In [ ]:
# Wikipedia statistics
print("\nWikipedia Text Statistics:")
text_lengths = [len(x['text']) for x in wiki_nl]
print(f"  Avg text length: {sum(text_lengths) / len(text_lengths):.0f} chars")
print(f"  Min text length: {min(text_lengths):,} chars")
print(f"  Max text length: {max(text_lengths):,} chars")
print(f"  Total chars: {sum(text_lengths):,}")

# Show some random titles
print(f"\nRandom article titles:")
import random
for i in random.sample(range(len(wiki_nl)), 5):
    print(f"  - {wiki_nl[i]['title']}")

## 2. MC4 Dutch

In [ ]:
# Load MC4 Dutch
mc4_nl = load_from_disk('mc4_nl')

print("=" * 60)
print("MC4 DUTCH (Common Crawl)")
print("=" * 60)
print(f"Total samples: {len(mc4_nl):,}")
print(f"Columns: {mc4_nl.column_names}")
print(f"Dataset size: ~4.58 GB")
print()
print("Sample document:")
print(f"URL: {mc4_nl[0]['url']}")
print(f"Language: {mc4_nl[0]['language']}")
print(f"Text preview (first 300 chars):\n{mc4_nl[0]['text'][:300]}...")

In [ ]:
# MC4 statistics
print("\nMC4 Text Statistics:")
mc4_text_lengths = [len(x['text']) for x in mc4_nl]
print(f"  Avg text length: {sum(mc4_text_lengths) / len(mc4_text_lengths):.0f} chars")
print(f"  Min text length: {min(mc4_text_lengths):,} chars")
print(f"  Max text length: {max(mc4_text_lengths):,} chars")
print(f"  Total chars: {sum(mc4_text_lengths):,}")

## 3. SQuAD NL (Question Answering)

In [ ]:
# Load SQuAD NL
squad_nl = load_from_disk('squad_nl')

print("=" * 60)
print("SQUAD NL (Question Answering)")
print("=" * 60)
print(f"Total samples: {len(squad_nl):,}")
print(f"Columns: {squad_nl.column_names}")
print(f"Dataset size: ~100 MB")
print()
print("Sample Q&A:")
sample = squad_nl[0]
print(f"Context: {sample['context'][:200]}...")
print(f"Question: {sample['question']}")
if sample['answers']['text']:
    print(f"Answer: {sample['answers']['text'][0]}")

## 4. Combined Statistics

In [ ]:
# Summary table
import pandas as pd

summary = pd.DataFrame({
    'Dataset': ['Wikipedia NL', 'MC4 Dutch', 'SQuAD NL'],
    'Size (GB)': [2.5, 4.58, 0.1],
    'Samples': [f"{len(wiki_nl):,}", f"{len(mc4_nl):,}", f"{len(squad_nl):,}"],
    'Type': ['Encyclopedic', 'Web Crawl', 'Q&A']
})

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(summary.to_string(index=False))
print()
print(f"Total size: {2.5 + 4.58 + 0.1:.2f} GB")
print(f"Total samples: {len(wiki_nl) + len(mc4_nl) + len(squad_nl):,}")

## 5. Text Preprocessing Example

In [ ]:
import re

def clean_text(text):
    """Clean Dutch text for LLM training"""
    # Remove extra whitespace
    text = ' '.join(text.split())
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove special characters but keep punctuation
    text = re.sub(r'[^\w\s.!?,;:\'"-]', '', text)
    return text.strip()

print("Text Cleaning Example:")
print("\nOriginal Wikipedia text:")
original = wiki_nl[100]['text'][:300]
print(original)

print("\nCleaned text:")
cleaned = clean_text(original)
print(cleaned)

## 6. Tokenization Example (Hugging Face Tokenizers)

In [ ]:
from transformers import AutoTokenizer

# Load Dutch tokenizer
tokenizer = AutoTokenizer.from_pretrained('pdelobelle/robbert-v2-dutch-base')

print("Tokenization Example:")
text = "Dit is een Nederlands LLM trainingset."

print(f"Original text: {text}")
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")
print(f"Token count: {len(tokens)}")

# Encode
encoded = tokenizer.encode(text, return_tensors='pt')
print(f"\nEncoded (token IDs): {encoded}")

## Next Steps

1. **Preprocessing** - Clean and tokenize datasets
2. **Data split** - Train/validation splits
3. **Model selection** - Choose base model (GPT-2, LLaMA, etc.)
4. **Training** - Fine-tune on Dutch data
5. **Evaluation** - Test model performance